# EEG_28 — cVAE Condizionale: Dati Sintetici EEG per Parola e Fenotipo

**Obiettivo**: addestrare un Conditional VAE (cVAE) che generi trial EEG sintetici  
condizionati su `(parola, cluster)` → più dati per analisi e augmentation in EEG_26/27.

**Architettura**:
```
ENCODER
  x (B,61,384) → CNN1D → flatten
  + cond(word_emb + cluster_onehot)
  → MLP → μ (B,64), log_var (B,64)

DECODER
  z (B,64) + cond → MLP → reshape (B,256,8)
  → Upsample+Conv × 4 → x̂ (B,61,384)
```

**Condizionamento**:
- `word_emb`: `nn.Embedding(110, 32)` — parola (0–109)
- `cluster_onehot`: 2 dim — C0 o C1
- `cond_dim = 34`

**Loss**: ELBO = MSE ricostruzione + β·KL  
β-VAE (β>1) per spazio latente più strutturato e separato per parola.

**Analisi incluse**:
- §6: confronto trial reale vs ricostruzione vs sintetico fresh (stessa parola, stesso cluster)
- §7: generazione batch sintetici + salvataggio su disco
- §8: UMAP dello spazio latente z colorato per parola / cluster / soggetto
- §9: distanze centroide inter-parola in z per C0 vs C1

In [ ]:
import json, logging, re
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import pairwise_distances
import wandb

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('eeg28')

project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'.git').exists()), Path.cwd())
FIG_DIR = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
SYN_DIR = project_root / 'data' / 'synthetic_cvae'; SYN_DIR.mkdir(exist_ok=True)

# ---- CONFIG ----
N_CHANNELS   = 61
N_SAMPLES    = 384
N_WORDS      = 110      # parole totali
N_CLUSTERS   = 2        # C0, C1
N_CONCR      = 4        # classi concr4 (per analisi)
CLUSTER_SCHEME = 'concr4'

# Split standard
SUBJ_TRAIN = list(range(0, 50))
SUBJ_VAL   = list(range(50, 60))
SUBJ_TEST  = list(range(60, 74))

# cVAE architettura
Z_DIM        = 64       # dimensione spazio latente
WORD_EMB_DIM = 32       # embedding parola
COND_DIM     = WORD_EMB_DIM + N_CLUSTERS   # 34
BETA         = 4.0      # β-VAE: >1 per spazio più separato

# Training
LR           = 1e-3
BATCH_SIZE   = 64
MAX_EPOCHS   = 80
PATIENCE     = 15
USE_INSTANCE_NORM = True
DATA_METRIC  = 'abs_pcc'

# Generazione
N_SYNTH_PER_WORD = 50   # trial sintetici per parola per cluster

WANDB_ENTITY  = 'uras-daniele22-politecnico-di-milano'
WANDB_PROJECT = 'miralis-imagined-speech'

_PAT = re.compile(r'^P(\d+)_S(\d+)$')
label2cluster_concr = {int(k): int(v) for k,v in json.loads(
    (project_root/'configs'/'label_schemes'/'labelid2cluster_concr4.json').read_text()).items()}
label2idx = json.loads((project_root/'configs'/'label_schemes'/'label2idx.json').read_text())
idx2label = {v: k for k, v in label2idx.items()}   # word_idx → stringa parola

log.info(f'Z_DIM={Z_DIM}  COND_DIM={COND_DIM}  BETA={BETA}')
log.info(f'SYN_DIR: {SYN_DIR}')

## §2 — Cluster labels C0/C1

In [ ]:
cluster_df  = pd.read_csv(project_root / 'figures' / 'eeg08c_subject_clusters.csv')
sid2pheno   = dict(zip(cluster_df.subj_id, cluster_df.cluster_k2))  # 0=C0, 1=C1
C0_ALL = sorted(cluster_df[cluster_df.cluster_k2 == 0].subj_id.tolist())
C1_ALL = sorted(cluster_df[cluster_df.cluster_k2 == 1].subj_id.tolist())
log.info(f'C0: {len(C0_ALL)} soggetti  C1: {len(C1_ALL)} soggetti')

## §3 — Dataset

Il dataset per il cVAE restituisce `(x, word_idx, cluster_idx)`.  
- `word_idx` (0–109): identifica la **parola** → usato per l'embedding condizionale  
- `cluster_idx` (0 o 1): C0/C1 del soggetto → usato per il one-hot condizionale

In [ ]:
class EEGcVAEDataset(Dataset):
    """
    Restituisce (x, word_idx, cluster_idx, subj_id).
    word_idx  : 0–109 — ID della parola (per l'embedding condizionale)
    cluster_idx: 0 o 1 — fenotipo del soggetto
    subj_id   : per analisi spazio latente
    """
    def __init__(self, subj_ids, metric=DATA_METRIC):
        root = project_root / 'data' / f'hypergraphs_pruned_{metric}'
        self.items = []  # (path, word_idx, cluster_idx, subj_id)
        for p in sorted(root.rglob('trial_*.pt')):
            m = _PAT.match(p.parent.name)
            if not m: continue
            sid = int(m.group(1))
            if sid not in subj_ids: continue
            cluster_idx = sid2pheno.get(sid)
            if cluster_idx is None: continue
            d = torch.load(p, weights_only=False)
            # y è il word_idx (0-109) — NON il concr4 cluster
            word_idx = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
            self.items.append((p, word_idx, cluster_idx, sid))
        log.info(f'  {len(self.items)} trial | {len(subj_ids)} soggetti')

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        p, word_idx, cluster_idx, sid = self.items[idx]
        d = torch.load(p, weights_only=False)
        x = d['x'].float()
        if USE_INSTANCE_NORM:
            x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)
        return (x,
                torch.tensor(word_idx,    dtype=torch.long),
                torch.tensor(cluster_idx, dtype=torch.long),
                torch.tensor(sid,         dtype=torch.long))


def make_loaders():
    tr = EEGcVAEDataset(SUBJ_TRAIN)
    va = EEGcVAEDataset(SUBJ_VAL)
    kw = dict(num_workers=2, pin_memory=True)
    return (DataLoader(tr, BATCH_SIZE, shuffle=True,  **kw),
            DataLoader(va, BATCH_SIZE, shuffle=False, **kw))

## §4 — Architettura cVAE

### Encoder CNN
```
x (B,61,384)
  Conv1d(61→128, k=8, s=4)  → (B,128,95)
  Conv1d(128→256, k=4, s=2) → (B,256,46)
  Conv1d(256→256, k=4, s=2) → (B,256,22)
  AdaptiveAvgPool1d(8)       → (B,256,8)
  Flatten                    → (B,2048)
  concat cond (B,34)         → (B,2082)
  Linear → μ, log_var        → (B,64) each
```

### Decoder CNN
```
z (B,64) concat cond (B,34)  → (B,98)
  Linear → (B,2048)
  Reshape → (B,256,8)
  Upsample×3 + Conv → (B,256,24)
  Upsample×2 + Conv → (B,256,48)
  Upsample×2 + Conv → (B,128,96)
  Upsample×4 + Conv → (B,61,384)
```

In [ ]:
class CondEmbedding(nn.Module):
    """Embedding condizionale: word (0-109) + cluster (0/1) → vettore cond (B, COND_DIM)."""
    def __init__(self):
        super().__init__()
        self.word_emb = nn.Embedding(N_WORDS, WORD_EMB_DIM)

    def forward(self, word_idx, cluster_idx):
        # word_idx: (B,) long  cluster_idx: (B,) long
        we = self.word_emb(word_idx)                                  # (B, 32)
        co = F.one_hot(cluster_idx, N_CLUSTERS).float()               # (B, 2)
        return torch.cat([we, co], dim=1)                             # (B, 34)


class CVAEEncoder(nn.Module):
    def __init__(self, cond_dim=COND_DIM, z_dim=Z_DIM):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(N_CHANNELS, 128, kernel_size=8, stride=4),
            nn.BatchNorm1d(128), nn.ELU(),
            nn.Conv1d(128, 256, kernel_size=4, stride=2),
            nn.BatchNorm1d(256), nn.ELU(),
            nn.Conv1d(256, 256, kernel_size=4, stride=2),
            nn.BatchNorm1d(256), nn.ELU(),
            nn.AdaptiveAvgPool1d(8),
        )
        flat_dim = 256 * 8  # 2048
        self.fc_mu      = nn.Linear(flat_dim + cond_dim, z_dim)
        self.fc_log_var = nn.Linear(flat_dim + cond_dim, z_dim)

    def forward(self, x, cond):
        h = self.cnn(x).flatten(1)          # (B, 2048)
        h = torch.cat([h, cond], dim=1)     # (B, 2048+34)
        return self.fc_mu(h), self.fc_log_var(h)


class CVAEDecoder(nn.Module):
    """
    Condizionato SOLO su cluster_onehot (2 dim), NON su word_emb.
    Conditioning asimmetrico: encoder vede (word+cluster), decoder vede solo cluster.
    Questo forza z a portare l'informazione sulla parola → previene posterior collapse.
    """
    def __init__(self, z_dim=Z_DIM):
        super().__init__()
        decoder_cond_dim = N_CLUSTERS  # solo 2 dim: cluster onehot
        self.fc = nn.Linear(z_dim + decoder_cond_dim, 256 * 8)
        # Upsample: 8 → 24 → 48 → 96 → 384
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=3),
            nn.Conv1d(256, 256, kernel_size=3, padding=1), nn.BatchNorm1d(256), nn.ELU(),
            nn.Upsample(scale_factor=2),
            nn.Conv1d(256, 256, kernel_size=3, padding=1), nn.BatchNorm1d(256), nn.ELU(),
            nn.Upsample(scale_factor=2),
            nn.Conv1d(256, 128, kernel_size=3, padding=1), nn.BatchNorm1d(128), nn.ELU(),
            nn.Upsample(scale_factor=4),
            nn.Conv1d(128, N_CHANNELS, kernel_size=3, padding=1),  # (B, 61, 384)
        )

    def forward(self, z, cluster_idx):
        co = F.one_hot(cluster_idx, N_CLUSTERS).float()    # (B, 2)
        h = self.fc(torch.cat([z, co], dim=1))             # (B, 2048)
        h = h.view(h.size(0), 256, 8)                      # (B, 256, 8)
        return self.up(h)                                  # (B, 61, 384)


class CVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.cond_emb = CondEmbedding()
        self.encoder  = CVAEEncoder()
        self.decoder  = CVAEDecoder()

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x, word_idx, cluster_idx):
        cond  = self.cond_emb(word_idx, cluster_idx)       # encoder vede word+cluster
        mu, log_var = self.encoder(x, cond)
        z     = self.reparameterize(mu, log_var)
        x_hat = self.decoder(z, cluster_idx)               # decoder vede solo cluster
        return x_hat, mu, log_var

    @torch.no_grad()
    def encode(self, x, word_idx, cluster_idx):
        cond = self.cond_emb(word_idx, cluster_idx)
        mu, _ = self.encoder(x, cond)
        return mu

    @torch.no_grad()
    def generate(self, word_idx, cluster_idx, n_samples=1, device='cpu'):
        """Campiona z ~ N(0,I) e decodifica → trial sintetici (n_samples, 61, 384)."""
        word_t    = torch.tensor([word_idx]    * n_samples, dtype=torch.long,  device=device)
        cluster_t = torch.tensor([cluster_idx] * n_samples, dtype=torch.long,  device=device)
        cond   = self.cond_emb(word_t, cluster_t)
        z      = torch.randn(n_samples, Z_DIM, device=device)
        return self.decoder(z, cluster_t)                  # (n_samples, 61, 384)


def elbo_loss(x, x_hat, mu, log_var, beta=1.0, free_bits=0.5):
    """
    ELBO con free bits per prevenire posterior collapse.

    free_bits: KL minimo garantito per dimensione latente (nats).
    Ogni dimensione z_i deve usare almeno free_bits nats → il decoder non può
    ignorare completamente z anche se aumenta β.

    Con Z_DIM=64 e free_bits=0.5 → KL_min_totale = 32 nats (media = 0.5).
    """
    recon = F.mse_loss(x_hat, x, reduction='mean')
    # KL per dimensione: (B, Z_DIM) → media sul batch → (Z_DIM,)
    kl_per_dim = -0.5 * (1 + log_var - mu.pow(2) - log_var.exp())  # (B, Z_DIM)
    kl_avg     = kl_per_dim.mean(0)                                  # (Z_DIM,)
    # Free bits: clamp → ogni dim usa almeno free_bits nats
    kl_clamped = torch.clamp(kl_avg, min=free_bits).mean()           # scalare
    kl_log     = kl_avg.mean()                                       # per logging (senza clamp)
    return recon + beta * kl_clamped, recon, kl_log


# Sanity check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
log.info(f'device: {device}')
_m  = CVAE().to(device)
_x  = torch.randn(4, N_CHANNELS, N_SAMPLES).to(device)
_wi = torch.randint(0, N_WORDS, (4,)).to(device)
_ci = torch.randint(0, N_CLUSTERS, (4,)).to(device)
_xh, _mu, _lv = _m(_x, _wi, _ci)
assert _xh.shape == (4, N_CHANNELS, N_SAMPLES), f'Shape errata: {_xh.shape}'
assert _mu.shape == (4, Z_DIM)
n_p = sum(p.numel() for p in _m.parameters() if p.requires_grad)
log.info(f'CVAE OK — {n_p:,} param  z_dim={Z_DIM}  decoder_cond=cluster_only(2dim)')
log.info('Conditioning asimmetrico: encoder(word+cluster) → z → decoder(cluster)')
del _m, _x, _wi, _ci, _xh, _mu, _lv

## §5 — Training

In [ ]:
CHECKPOINT_CVAE = project_root / 'data' / 'eeg28_cvae_checkpoint.pt'
FORCE_RETRAIN   = True

FREE_BITS = 0.5   # KL minimo per dimensione latente (nats) — previene posterior collapse

def run_epoch_vae(model, loader, optimizer=None, beta=1.0, free_bits=FREE_BITS):
    train = optimizer is not None
    model.train() if train else model.eval()
    tot_loss, tot_recon, tot_kl, n = 0.0, 0.0, 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, word_idx, cluster_idx, _ in loader:
            x, word_idx, cluster_idx = x.to(device), word_idx.to(device), cluster_idx.to(device)
            x_hat, mu, log_var = model(x, word_idx, cluster_idx)
            loss, recon, kl = elbo_loss(x, x_hat, mu, log_var, beta=beta, free_bits=free_bits)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            b = len(x)
            tot_loss  += loss.item()  * b
            tot_recon += recon.item() * b
            tot_kl    += kl.item()   * b
            n += b
    return tot_loss/n, tot_recon/n, tot_kl/n


if CHECKPOINT_CVAE.exists() and not FORCE_RETRAIN:
    log.info(f'Carico checkpoint da {CHECKPOINT_CVAE}')
    model = CVAE().to(device)
    model.load_state_dict(torch.load(CHECKPOINT_CVAE, map_location=device))
    log.info('Checkpoint caricato OK')
else:
    if FORCE_RETRAIN and CHECKPOINT_CVAE.exists():
        CHECKPOINT_CVAE.unlink()
        log.info('Checkpoint rimosso — ripartenza da zero')

    BETA_MAX      = 1.0
    ANNEAL_EPOCHS = 20

    log.info(f'Avvio training cVAE: KL annealing β 0→{BETA_MAX} in {ANNEAL_EPOCHS} epoche')
    log.info(f'Free bits={FREE_BITS} nats/dim — conditioning asimmetrico (decoder=cluster only)')
    tr_l, va_l = make_loaders()
    log.info(f'  train={len(tr_l.dataset)}  val={len(va_l.dataset)}')

    model = CVAE().to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)

    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                     name=f'eeg28_cVAE_asym_fb{FREE_BITS}',
                     config=dict(notebook='EEG_28', model='cVAE_asymmetric',
                                 z_dim=Z_DIM, cond_dim=COND_DIM,
                                 beta_max=BETA_MAX, anneal_epochs=ANNEAL_EPOCHS,
                                 free_bits=FREE_BITS,
                                 decoder_cond='cluster_only_2dim',
                                 word_emb_dim=WORD_EMB_DIM,
                                 lr=LR, batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
                                 n_words=N_WORDS, n_clusters=N_CLUSTERS,
                                 n_train_subj=len(SUBJ_TRAIN)),
                     reinit='finish_previous',
                     settings=wandb.Settings(start_method='thread'))

    for epoch in range(1, MAX_EPOCHS + 1):
        beta_now = min(BETA_MAX, (epoch / ANNEAL_EPOCHS) * BETA_MAX)

        tr_loss, tr_r, tr_k = run_epoch_vae(model, tr_l, opt, beta=beta_now)
        va_loss, va_r, va_k = run_epoch_vae(model, va_l,       beta=beta_now)
        sched.step()

        run.log({'train/loss':  tr_loss, 'train/recon': tr_r, 'train/kl': tr_k,
                 'val/loss':    va_loss, 'val/recon':   va_r, 'val/kl':   va_k,
                 'beta':        beta_now, 'epoch': epoch})

        if epoch % 10 == 0:
            log.info(f'  epoch {epoch:3d}  β={beta_now:.2f}: '
                     f'val_loss={va_loss:.5f}  recon={va_r:.5f}  kl={va_k:.5f}')

    final_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    torch.save(final_state, CHECKPOINT_CVAE)
    run.summary['final_val_recon'] = va_r
    run.summary['final_val_kl']    = va_k
    run.finish()
    log.info(f'Training done. final recon={va_r:.5f}  kl={va_k:.5f}. Salvato in {CHECKPOINT_CVAE}')

## §6 — Confronto: Reale vs Ricostruzione vs Sintetico

Per un soggetto di test (default: P070, miglior C1):  
1. Prende N trial reali di una parola concreta (es. 'acqua')  
2. Li passa attraverso l'encoder → decoder → **ricostruzione**  
3. Campiona z ~ N(0,I) → decoder → **sintetico fresh** (stessa parola, stesso cluster)  
4. Plot: waveform canali frontali + occipitali + topomap media

In [ ]:
model.eval()

# --- Soggetto e parola da analizzare ---
TARGET_SID  = 70     # P070 — miglior soggetto C1
TARGET_WORD = 'acqua'  # parola concreta
TARGET_WORD_IDX = label2idx[TARGET_WORD]  # int
TARGET_CLUSTER  = sid2pheno[TARGET_SID]   # 1 = C1

log.info(f'Target: P{TARGET_SID:03d} | "{TARGET_WORD}" (word_idx={TARGET_WORD_IDX}) | cluster={TARGET_CLUSTER}')

# Carica i trial reali del soggetto target per la parola target
root_data = project_root / 'data' / f'hypergraphs_pruned_{DATA_METRIC}'
real_trials = []
for p in sorted(root_data.rglob('trial_*.pt')):
    m = _PAT.match(p.parent.name)
    if not m or int(m.group(1)) != TARGET_SID: continue
    d = torch.load(p, weights_only=False)
    word_idx = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
    if word_idx != TARGET_WORD_IDX: continue
    x = d['x'].float()
    if USE_INSTANCE_NORM:
        x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)
    real_trials.append(x)

log.info(f'Trial reali trovati: {len(real_trials)}')
if not real_trials:
    raise RuntimeError(f'Nessun trial trovato per P{TARGET_SID:03d} + "{TARGET_WORD}"')

X_real = torch.stack(real_trials)  # (N_real, 61, 384)

# --- Ricostruzione (encoder → decoder) ---
wi_t = torch.tensor([TARGET_WORD_IDX] * len(X_real), dtype=torch.long,  device=device)
ci_t = torch.tensor([TARGET_CLUSTER]  * len(X_real), dtype=torch.long,  device=device)
with torch.no_grad():
    X_recon, _, _ = model(X_real.to(device), wi_t, ci_t)
X_recon = X_recon.cpu()

# --- Sintetici fresh (z ~ N(0,I)) ---
with torch.no_grad():
    X_synth = model.generate(TARGET_WORD_IDX, TARGET_CLUSTER,
                              n_samples=len(X_real), device=device).cpu()

log.info(f'X_real:  {X_real.shape}  X_recon: {X_recon.shape}  X_synth: {X_synth.shape}')

# --- Plot ---
# Canali di riferimento
FRONTAL  = [0, 1, 2, 3]   # F3, F4, Fz, FC3 (circa)
OCCIPITAL = [50, 51, 52, 53]  # O1, O2, Oz, PO7 (circa)
t = np.linspace(0, 384/256, 384)  # asse temporale in secondi

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle(f'EEG_28 — cVAE: Reale vs Ricostruzione vs Sintetico\n'
             f'P{TARGET_SID:03d} ({"C0" if TARGET_CLUSTER==0 else "C1"}) | "{TARGET_WORD}"',
             fontsize=13, fontweight='bold')

COLORS = {'reale': '#1565C0', 'ricostr': '#E65100', 'sintetico': '#2E7D32'}

for row, (data, label, color) in enumerate([
    (X_real,  'Reale',         COLORS['reale']),
    (X_recon, 'Ricostruzione', COLORS['ricostr']),
    (X_synth, 'Sintetico fresh', COLORS['sintetico']),
]):
    mean_trial = data.mean(0).numpy()  # (61, 384)

    # Waveform frontali
    ax1 = fig.add_subplot(gs[row, 0])
    for ch in FRONTAL:
        ax1.plot(t, mean_trial[ch], alpha=0.7, lw=1)
    ax1.set_title(f'{label} — Canali frontali')
    ax1.set_xlabel('s'); ax1.set_ylabel('μV (norm)')
    ax1.axhline(0, color='gray', ls=':', lw=0.8)

    # Waveform occipitali
    ax2 = fig.add_subplot(gs[row, 1])
    for ch in OCCIPITAL:
        ax2.plot(t, mean_trial[ch], alpha=0.7, lw=1, color=color)
    ax2.set_title(f'{label} — Canali occipitali')
    ax2.set_xlabel('s'); ax2.set_ylabel('μV (norm)')
    ax2.axhline(0, color='gray', ls=':', lw=0.8)

    # Topomap approssimata (RMS per canale come proxy potenza)
    ax3 = fig.add_subplot(gs[row, 2])
    rms_per_ch = np.sqrt((mean_trial ** 2).mean(axis=1))  # (61,)
    im = ax3.imshow(rms_per_ch.reshape(1, -1), aspect='auto', cmap='RdBu_r',
                    vmin=0, vmax=rms_per_ch.max())
    ax3.set_title(f'{label} — RMS per canale')
    ax3.set_xlabel('Canale (0–60)'); ax3.set_yticks([])
    plt.colorbar(im, ax=ax3, shrink=0.8)

plt.savefig(FIG_DIR / 'eeg28_reale_vs_sintetico.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Metriche quantitative ---
mse_recon  = F.mse_loss(X_recon, X_real).item()
mse_synth  = F.mse_loss(X_synth, X_real).item()
corr_recon = np.corrcoef(X_real.flatten(), X_recon.flatten())[0,1]
corr_synth = np.corrcoef(X_real.flatten(), X_synth.flatten())[0,1]
print(f'\nMSE  Reale↔Ricostruzione: {mse_recon:.6f}')
print(f'MSE  Reale↔Sintetico:     {mse_synth:.6f}')
print(f'Corr Reale↔Ricostruzione: {corr_recon:.4f}')
print(f'Corr Reale↔Sintetico:     {corr_synth:.4f}')
print('\nNota: il sintetico è campionato da N(0,I) — bassa corr attesa ma struttura simile')

### §6b — Confronto distribuzioni spettrali

Il VAE deve preservare la struttura spettrale del segnale EEG (picco alpha ~10Hz, ecc.).

In [ ]:
from scipy import signal as scipy_signal

FS = 256  # Hz
CH_PLOT = 0  # primo canale (frontale)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
fig.suptitle(f'Densità Spettrale di Potenza — P{TARGET_SID:03d} | "{TARGET_WORD}"', fontsize=12)

for ax, (data, label, color) in zip(axes, [
    (X_real,  'Reale',         COLORS['reale']),
    (X_recon, 'Ricostruzione', COLORS['ricostr']),
    (X_synth, 'Sintetico',     COLORS['sintetico']),
]):
    # PSD per ogni trial, poi media
    psds = []
    for trial_idx in range(len(data)):
        sig = data[trial_idx, CH_PLOT].numpy()
        f, psd = scipy_signal.welch(sig, fs=FS, nperseg=128)
        psds.append(psd)
    psds = np.array(psds)
    psd_mean = psds.mean(0)
    psd_std  = psds.std(0)
    ax.semilogy(f, psd_mean, color=color, lw=2, label=label)
    ax.fill_between(f, psd_mean - psd_std, psd_mean + psd_std, alpha=0.2, color=color)
    # Bande EEG
    for band, (lo, hi, bcolor) in {'delta':(1,4,'#90CAF9'), 'theta':(4,8,'#A5D6A7'),
                                    'alpha':(8,13,'#FFCC80'), 'beta':(13,30,'#EF9A9A')}.items():
        ax.axvspan(lo, hi, alpha=0.08, color=bcolor, label=band)
    ax.set_xlabel('Frequenza (Hz)')
    ax.set_title(label)
    ax.set_xlim(1, 50)
    if ax == axes[0]: ax.set_ylabel('PSD (μV²/Hz)')

axes[0].legend(fontsize=7, loc='upper right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg28_psd_confronto.png', dpi=150, bbox_inches='tight')
plt.show()

## §7 — Generazione batch sintetici

Genera `N_SYNTH_PER_WORD` trial per ogni (parola × cluster).  
Totale: 110 × 2 × 50 = 11,000 trial sintetici.  
Salva come `data/synthetic_cvae/{cluster}/word_{word_idx:03d}.pt`.

In [ ]:
model.eval()
log.info(f'Generazione {N_WORDS} × {N_CLUSTERS} × {N_SYNTH_PER_WORD} trial sintetici...')

for cluster_id in range(N_CLUSTERS):
    cluster_name = f'C{cluster_id}'
    (SYN_DIR / cluster_name).mkdir(exist_ok=True)
    for word_idx in range(N_WORDS):
        word_str = idx2label[word_idx]
        with torch.no_grad():
            x_synth = model.generate(word_idx, cluster_id,
                                      n_samples=N_SYNTH_PER_WORD,
                                      device=device).cpu()  # (N_SYNTH, 61, 384)
        save_path = SYN_DIR / cluster_name / f'word_{word_idx:03d}_{word_str}.pt'
        torch.save({'x': x_synth, 'word_idx': word_idx, 'word': word_str, 'cluster': cluster_id}, save_path)
    log.info(f'  {cluster_name}: {N_WORDS} file salvati in {SYN_DIR / cluster_name}')

total_synth = N_WORDS * N_CLUSTERS * N_SYNTH_PER_WORD
log.info(f'Generazione completata. Totale trial sintetici: {total_synth:,}')

## §8 — Analisi spazio latente z (UMAP)

Codifica tutti i trial di test → μ (B, 64) → UMAP 2D.  
Plot 1: colorato per **parola** (vedi se le parole si separano)  
Plot 2: colorato per **cluster** C0/C1  
Plot 3: colorato per **classe concr4** (più interpretabile di 110 parole)

In [ ]:
try:
    import umap
    UMAP_AVAIL = True
except ImportError:
    from sklearn.manifold import TSNE
    UMAP_AVAIL = False
    log.warning('umap-learn non disponibile, uso t-SNE. Installa con: pip install umap-learn')

model.eval()
log.info('Codifica trial di test nello spazio latente...')

te_ds = EEGcVAEDataset(SUBJ_TEST)
te_loader = DataLoader(te_ds, BATCH_SIZE, shuffle=False, num_workers=2)

all_mu, all_word, all_cluster, all_subj = [], [], [], []
with torch.no_grad():
    for x, word_idx, cluster_idx, sid in te_loader:
        mu = model.encode(x.to(device), word_idx.to(device), cluster_idx.to(device))
        all_mu.extend(mu.cpu().numpy())
        all_word.extend(word_idx.numpy())
        all_cluster.extend(cluster_idx.numpy())
        all_subj.extend(sid.numpy())

Z   = np.array(all_mu)       # (N_test, 64)
W   = np.array(all_word)     # word_idx 0-109
C   = np.array(all_cluster)  # 0=C0, 1=C1
S   = np.array(all_subj)     # subject id
# Classe concr4 di ogni trial
CC  = np.array([label2cluster_concr.get(int(w), -1) for w in W])
log.info(f'Z shape: {Z.shape}')

# --- Riduzione dimensionale ---
if UMAP_AVAIL:
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
    Z2 = reducer.fit_transform(Z)
    method = 'UMAP'
else:
    Z2 = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(Z)
    method = 't-SNE'
log.info(f'{method} completato. Z2 shape: {Z2.shape}')

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'EEG_28 — Spazio latente z ({method}) — soggetti test', fontsize=13, fontweight='bold')

# Plot 1: per cluster C0/C1
ax = axes[0]
for ci, (cname, ccolor) in enumerate([("C0", "#EF5350"), ("C1", "#42A5F5")]):
    mask = C == ci
    ax.scatter(Z2[mask, 0], Z2[mask, 1], c=ccolor, s=4, alpha=0.5, label=cname)
ax.set_title('Colorato per fenotipo (C0/C1)')
ax.legend(markerscale=3, fontsize=9)
ax.set_xlabel(method+' 1'); ax.set_ylabel(method+' 2')

# Plot 2: per classe concr4
ax2 = axes[1]
concr_names = ['Concreto', 'Azione', 'Stato', 'Astratto']
colors4 = ['#1565C0', '#E65100', '#2E7D32', '#6A1B9A']
for ci, (cname, ccolor) in enumerate(zip(concr_names, colors4)):
    mask = CC == ci
    ax2.scatter(Z2[mask, 0], Z2[mask, 1], c=ccolor, s=4, alpha=0.5, label=cname)
ax2.set_title('Colorato per classe concr4')
ax2.legend(markerscale=3, fontsize=8)
ax2.set_xlabel(method+' 1')

# Plot 3: per soggetto
ax3 = axes[2]
subj_ids_uniq = sorted(set(S))
cmap = plt.cm.tab20
for i, sid in enumerate(subj_ids_uniq):
    mask = S == sid
    phen = sid2pheno.get(int(sid), -1)
    marker = 'o' if phen == 1 else '^'
    ax3.scatter(Z2[mask, 0], Z2[mask, 1], c=[cmap(i/len(subj_ids_uniq))],
                s=8, alpha=0.6, marker=marker)
ax3.set_title('Colorato per soggetto (○=C1, △=C0)')
ax3.set_xlabel(method+' 1')

plt.tight_layout()
plt.savefig(FIG_DIR / f'eeg28_latent_{method.lower()}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nSe le parole si separano in {method}: il VAE ha imparato la struttura semantica.')
print('Se i cluster si separano: il condizionamento C0/C1 è codificato in z.')
print('Se i soggetti si separano: lo spazio z è ancora influenzato dall\'identità soggetto (da minimizzare).')

## §9 — Distanze inter-parola in z: C0 vs C1

Domanda chiave: **le parole sono più separate nello spazio latente z per C0 o per C1?**

Se C1 ha distanze inter-parola maggiori → il cVAE ha imparato che in C1 le parole  
producono pattern EEG più distinti → coerente con C1 come fenotipo più decodificabile.

In [ ]:
from scipy.spatial.distance import cdist

def word_separation_score(Z_cluster, W_cluster):
    """
    Distanza media inter-parola vs intra-parola nello spazio z.
    Restituisce: delta = mean_inter - mean_intra (>0 = parole separate)
    """
    words_uniq = np.unique(W_cluster)
    centroids  = np.array([Z_cluster[W_cluster == w].mean(0) for w in words_uniq
                           if (W_cluster == w).sum() > 0])
    if len(centroids) < 2:
        return np.nan, np.nan, np.nan
    # Distanze inter-centroide
    D = cdist(centroids, centroids, metric='euclidean')
    mask_upper = np.triu(np.ones_like(D, dtype=bool), k=1)
    inter_dist = D[mask_upper].mean()
    # Varianza intra-parola (spread intorno al centroide)
    intra_vars = []
    for wi, w in enumerate(words_uniq):
        pts = Z_cluster[W_cluster == w]
        if len(pts) > 1:
            intra_vars.append(np.linalg.norm(pts - centroids[wi], axis=1).mean())
    intra_dist = np.mean(intra_vars)
    return inter_dist, intra_dist, inter_dist / (intra_dist + 1e-8)  # silhouette proxy


for cluster_id, cluster_name in [(0, 'C0'), (1, 'C1')]:
    mask = C == cluster_id
    Z_c  = Z[mask]
    W_c  = W[mask]
    inter, intra, ratio = word_separation_score(Z_c, W_c)
    print(f'{cluster_name} (n={mask.sum()}): inter={inter:.4f}  intra={intra:.4f}  ratio={ratio:.3f}')

print('\nRatio > 1: le parole sono separate (inter > intra).')
print('Ratio C1 > Ratio C0: C1 ha parole più distinte nello spazio z — più decodificabile.')

# --- Plot distribuzioni distanze per le 110 parole ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('EEG_28 — Separazione parole nello spazio latente z', fontsize=12, fontweight='bold')

for ax, (cluster_id, cluster_name, color) in zip(axes, [
    (0, 'C0', '#EF5350'), (1, 'C1', '#42A5F5')
]):
    mask = C == cluster_id
    Z_c, W_c = Z[mask], W[mask]
    words_uniq = np.unique(W_c)
    centroids  = np.array([Z_c[W_c == w].mean(0) for w in words_uniq if (W_c == w).sum() > 0])
    if len(centroids) < 2: continue
    D = cdist(centroids, centroids, metric='euclidean')
    inter_dists = D[np.triu(np.ones_like(D, dtype=bool), k=1)]
    ax.hist(inter_dists, bins=40, color=color, alpha=0.8, edgecolor='none')
    ax.axvline(inter_dists.mean(), color='black', ls='--', lw=1.5,
               label=f'Media: {inter_dists.mean():.3f}')
    ax.set_title(f'{cluster_name} — Distanze inter-centroide parole')
    ax.set_xlabel('Distanza euclidea in z'); ax.set_ylabel('Conteggio coppie parola')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg28_word_distances_z.png', dpi=150, bbox_inches='tight')
plt.show()

## §10 — Riepilogo e log W&B

In [ ]:
run_s = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                   name=f'eeg28_cVAE_analysis_beta{BETA}',
                   config=dict(notebook='EEG_28', model='cVAE', z_dim=Z_DIM,
                               cond_dim=COND_DIM, beta=BETA, n_synth_per_word=N_SYNTH_PER_WORD),
                   reinit='finish_previous',
                   settings=wandb.Settings(start_method='thread'))

run_s.summary['mse_reconstruction'] = mse_recon
run_s.summary['mse_synthetic']       = mse_synth
run_s.summary['corr_reconstruction'] = corr_recon
run_s.summary['corr_synthetic']      = corr_synth
run_s.summary['total_synth_trials']  = N_WORDS * N_CLUSTERS * N_SYNTH_PER_WORD

for img_path in [
    FIG_DIR / 'eeg28_reale_vs_sintetico.png',
    FIG_DIR / 'eeg28_psd_confronto.png',
    FIG_DIR / f'eeg28_latent_{method.lower()}.png',
    FIG_DIR / 'eeg28_word_distances_z.png',
]:
    if img_path.exists():
        run_s.log({img_path.stem: wandb.Image(str(img_path))})

run_s.finish()
print('W&B log completato.')
print(f'\nDati sintetici salvati in: {SYN_DIR}')
print('Usabili come augmentation in EEG_26 e EEG_27.')

## §11 — Note operative

**Come usare i sintetici in EEG_26/27**:
```python
# Caricare trial sintetici per una parola
d = torch.load('data/synthetic_cvae/C1/word_001_acqua.pt')
x_synth   = d['x']       # (50, 61, 384)
word_idx  = d['word_idx'] # 1
cluster   = d['cluster']  # 1
```

**Interpretazione §8 (UMAP)**:
- Se i punti si mescolano per parola → il VAE non ha separato le parole nello spazio z  
  (atteso: ε²(parola)=0.03 rende difficile anche per il VAE)
- Se C0 e C1 si separano → il condizionamento cluster è efficace
- Se i soggetti si separano → lo spazio z è ancora parzialmente soggetto-specifico

**β ottimale**: se la ricostruzione è pessima → abbassa β (prova β=1 o β=2).  
Se lo spazio z è caotico (UMAP informe) → aumenta β (prova β=8).